<a href="https://colab.research.google.com/github/flahbocchino/cardioia-fase2-diagnostico/blob/main/cardioia_parte1_extracao_sintomas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🫀 CardioIA — Fase 2
## Parte 1: Extração de Sintomas e Sugestão de Diagnóstico

**Aluna:** Flavia Nunes Bocchino  
**RM:** 564213  
**Curso:** Inteligencia Artificial - FIAP  

---

### O que este notebook faz?

Este codigo simula o funcionamento de um sistema basico de apoio ao diagnostico medico.

Ele realiza 3 etapas:
1. **Le** um arquivo de texto com 10 frases de pacientes descrevendo seus sintomas
2. **Identifica** os sintomas presentes em cada frase, comparando com um mapa de conhecimento
3. **Sugere** um diagnostico possivel com base nos sintomas encontrados

> **Importante:** Este sistema e academico e nao substitui avaliacao medica profissional.

In [3]:
# CELULA 1 - Importacao das bibliotecas necessarias
# Pandas: trabalha com tabelas e arquivos CSV
# Unicodedata e re: limpam o texto removendo acentos e pontuacao

import pandas as pd
import unicodedata
import re

print("Bibliotecas carregadas com sucesso!")

Bibliotecas carregadas com sucesso!


In [4]:
# CELULA 2 - Criacao dos arquivos no Colab

frases = """Ha dois dias estou sentindo uma dor forte no peito que aperta como se algo estivesse me comprimindo, e a dor piora muito quando subo escadas ou faco qualquer esforco fisico.
Sinto cansaco constante ha mais de uma semana, mesmo depois de dormir a noite toda fico exausto, e percebo que minhas pernas e tornozelos estao inchados no final do dia.
Tenho sentido falta de ar repentina mesmo em repouso, como se nao conseguisse completar uma respiracao fundo, e isso vem acontecendo varias vezes ao dia nos ultimos tres dias.
Meu coracao dispara do nada, bate muito rapido e de forma irregular por alguns minutos, me deixa tonto e com sensacao de desmaio, e isso ja aconteceu quatro vezes essa semana.
Estou com dor de cabeca muito forte e visao embacada ha dois dias, minha pressao estava 180x110 quando medi em casa, e sinto uma pressao na nuca que nao passa nem com analgesico.
Nos ultimos cinco dias tenho sentido um formigamento que comeca no braco esquerdo e sobe ate o pescoco, junto com uma sensacao de peso no peito que aparece principalmente de manha.
Sinto tonturas frequentes quando me levanto rapido, ja desmaiei uma vez na semana passada, e percebo que meu coracao as vezes pula uma batida e volta a bater forte logo em seguida.
Tenho acordado no meio da noite com falta de ar e preciso sentar na cama para conseguir respirar melhor, isso acontece quase toda noite ha uma semana e estou muito preocupado.
Sinto um cansaco fora do normal ao caminhar pouco mais de um quarteirao, minha respiracao fica ofegante rapido e sinto um desconforto leve no peito que passa quando paro de me movimentar.
Estou com nausea, suor frio e uma sensacao estranha de pressao no centro do peito desde esta manha, a dor irradia para meu braco esquerdo e me sinto muito fraco e ansioso."""

with open("sintomas.txt", "w", encoding="utf-8") as f:
    f.write(frases)

print("Arquivo sintomas.txt criado!")

mapa_csv = """sintoma1,sintoma2,sintoma3,doenca_associada,nivel_risco,descricao
dor no peito,aperto no torax,dor irradiando para o braco,Infarto Agudo do Miocardio,alto,Bloqueio subito de arteria coronaria com risco de morte
dor no peito,pressao no peito,desconforto toracico,Angina Estavel,medio,Dor por reducao temporaria do fluxo sanguineo ao coracao
cansaco constante,fadiga extrema,pernas inchadas,Insuficiencia Cardiaca,alto,Coracao nao consegue bombear sangue suficiente ao corpo
falta de ar,dificuldade para respirar,respiracao ofegante,Edema Pulmonar,alto,Acumulo de liquido nos pulmoes por falha cardiaca
palpitacoes,coracao acelerado,batimento irregular,Fibrilacao Atrial,alto,Arritmia grave com risco de AVC e falencia cardiaca
tontura,desmaio,perda de consciencia,Sincope Cardiaca,alto,Perda temporaria de consciencia por queda do debito cardiaco
pressao alta,dor de cabeca forte,visao embacada,Hipertensao Grave,alto,Pressao arterial elevada com risco de AVC e infarto
formigamento no braco,dormencia no braco esquerdo,peso no peito,Infarto Agudo do Miocardio,alto,Irradiacao para membros e sinal classico de infarto
suor frio,nausea,fraqueza subita,Infarto Agudo do Miocardio,alto,Sintomas atipicos de infarto mais comuns em mulheres e diabeticos
falta de ar ao deitar,acordar com falta de ar,tosse noturna,Insuficiencia Cardiaca Congestiva,alto,Liquido nos pulmoes que piora em posicao horizontal
batimento acelerado,coracao disparado,ansiedade com palpitacoes,Taquicardia Supraventricular,medio,Ritmo cardiaco acelerado originado acima dos ventriculos
dor no peito ao esforco,dor que melhora em repouso,pressao no torax,Angina de Esforco,medio,Isquemia miocardica desencadeada por atividade fisica
inchaço nas pernas,tornozelos inchados,ganho de peso rapido,Insuficiencia Cardiaca,alto,Retencao de liquidos por falha no bombeamento cardiaco
pulso irregular,coracao pulando batidas,sensacao de falha no peito,Extrassistole,baixo,Batimentos extras geralmente benignos mas que exigem avaliacao
pressao baixa,tontura ao levantar,fraqueza nas pernas,Hipotensao Ortostatica,baixo,Queda de pressao ao mudar de posicao podendo indicar problema cardiaco
cansaco aos pequenos esforcos,falta de ar ao caminhar,limitacao fisica progressiva,Cardiomiopatia Dilatada,alto,Coracao dilatado com funcao de bombeamento reduzida
dor nas costas irradiando para o peito,dor intensa e rasgando,pressao muito alta,Disseccao Aortica,alto,Emergencia com ruptura da parede da aorta com risco de morte imediato
febre,dor no peito que piora ao respirar,cansaco intenso,Pericardite,medio,Inflamacao do saco que envolve o coracao geralmente por infeccao viral
confusao mental,dificuldade de falar,fraqueza em um lado do corpo,AVC Cardioembólico,alto,Coagulo do coracao que bloqueia arteria cerebral
chiado no peito,tosse com catarro,falta de ar cronica,Cor Pulmonale,alto,Falencia do lado direito do coracao causada por doenca pulmonar cronica
dor no peito que irradia para o pescoco,sudorese,mal estar geral,Infarto Agudo do Miocardio,alto,Irradiacao para pescoco e mandibula e sinal classico de infarto
coracao lento,cansaco sem motivo,sensacao de desmaiando,Bradicardia,medio,Frequencia cardiaca muito baixa que pode causar tontura e sincope
palidez,labios azulados,extremidades frias,Choque Cardiogenico,alto,Falencia grave do coracao com risco iminente de morte
pressao no peito apos refeicao,desconforto pos-prandial,queimacao no peito,Angina Variante,medio,Espasmo coronario que pode ocorrer em repouso ou apos refeicoes
historico de infarto,dor no peito recorrente,uso de nitrato,Angina Instavel,alto,Precursora de infarto com padrao de dor em piora progressiva"""

with open("mapa_conhecimento.csv", "w", encoding="utf-8") as f:
    f.write(mapa_csv)

print("Arquivo mapa_conhecimento.csv criado!")

Arquivo sintomas.txt criado!
Arquivo mapa_conhecimento.csv criado!


In [5]:
# CELULA 3 - Funcao para normalizar texto
# Normalizar significa padronizar o texto para comparacao:
# tudo minusculo, sem acentos, sem pontuacao

def normalizar(texto):
    texto = texto.lower()
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'[^\w\s]', ' ', texto)
    return texto

print("Funcao de normalizacao criada!")
print("Exemplo:")
print("  Original:    'Dor no Peito, Falta de Ar!'")
print("  Normalizado: '" + normalizar('Dor no Peito, Falta de Ar!') + "'")

Funcao de normalizacao criada!
Exemplo:
  Original:    'Dor no Peito, Falta de Ar!'
  Normalizado: 'dor no peito  falta de ar '


In [6]:
# CELULA 4 - Carregar arquivos e criar funcao de diagnostico

with open("sintomas.txt", "r", encoding="utf-8") as f:
    frases_pacientes = [linha.strip() for linha in f.readlines() if linha.strip()]

print(str(len(frases_pacientes)) + " frases de pacientes carregadas!")

mapa = pd.read_csv("mapa_conhecimento.csv")
print("Mapa de conhecimento carregado com " + str(len(mapa)) + " doencas mapeadas!")

def diagnosticar(frase, mapa):
    frase_norm = normalizar(frase)
    melhor_match = None
    maior_pontuacao = 0

    for _, linha in mapa.iterrows():
        pontuacao = 0
        sintomas_encontrados = []

        for col in ['sintoma1', 'sintoma2', 'sintoma3']:
            sintoma = normalizar(str(linha[col]))
            palavras_sintoma = sintoma.split()
            palavras_encontradas = [p for p in palavras_sintoma if p in frase_norm and len(p) > 3]
            if len(palavras_encontradas) >= 1:
                pontuacao += 1
                sintomas_encontrados.append(linha[col])

        if pontuacao > maior_pontuacao:
            maior_pontuacao = pontuacao
            melhor_match = {
                'doenca': linha['doenca_associada'],
                'risco': linha['nivel_risco'],
                'descricao': linha['descricao'],
                'sintomas_detectados': sintomas_encontrados,
                'pontuacao': pontuacao
            }

    return melhor_match

print("Funcao de diagnostico criada!")

10 frases de pacientes carregadas!
Mapa de conhecimento carregado com 25 doencas mapeadas!
Funcao de diagnostico criada!


In [7]:
# CELULA 5 - Executar o diagnostico para cada paciente

print("=" * 65)
print("     CARDIOIA - SISTEMA DE APOIO AO DIAGNOSTICO")
print("=" * 65)

emoji_risco = {'alto': 'ALTO', 'medio': 'MEDIO', 'baixo': 'BAIXO'}

for i, frase in enumerate(frases_pacientes, 1):
    resultado = diagnosticar(frase, mapa)

    print("\nPACIENTE " + str(i))
    print("Relato: " + frase[:80] + "...")

    if resultado and resultado['pontuacao'] > 0:
        risco = resultado['risco']
        print("Sintomas detectados: " + ', '.join(resultado['sintomas_detectados']))
        print("Diagnostico sugerido: " + resultado['doenca'])
        print("Nivel de risco: " + risco.upper())
        print("Descricao: " + resultado['descricao'])
    else:
        print("Nenhum sintoma identificado no mapa de conhecimento.")

    print("-" * 65)

print("\nAnalise concluida!")
print("Lembrete: Este sistema e academico. Sempre consulte um medico.")

     CARDIOIA - SISTEMA DE APOIO AO DIAGNOSTICO

PACIENTE 1
Relato: Ha dois dias estou sentindo uma dor forte no peito que aperta como se algo estiv...
Sintomas detectados: dor no peito, pressao no peito
Diagnostico sugerido: Angina Estavel
Nivel de risco: MEDIO
Descricao: Dor por reducao temporaria do fluxo sanguineo ao coracao
-----------------------------------------------------------------

PACIENTE 2
Relato: Sinto cansaco constante ha mais de uma semana, mesmo depois de dormir a noite to...
Sintomas detectados: cansaco constante, pernas inchadas
Diagnostico sugerido: Insuficiencia Cardiaca
Nivel de risco: ALTO
Descricao: Coracao nao consegue bombear sangue suficiente ao corpo
-----------------------------------------------------------------

PACIENTE 3
Relato: Tenho sentido falta de ar repentina mesmo em repouso, como se nao conseguisse co...
Sintomas detectados: falta de ar, respiracao ofegante
Diagnostico sugerido: Edema Pulmonar
Nivel de risco: ALTO
Descricao: Acumulo de liquid